In [ ]:
# 01 · SMOL 공동 학습 설정 (Easy + Medium + Hard RGB)
CFG = {
    'backend': 'smol',
    'run_name': 'moveboxes_smolvla_rgb_v1',
    'micro_batch': 1,
    'accumulate': 3,
    'chunk': 8,
    'execute_steps': 2,
    'inference_steps': 5,
    'image_size': 256,
    'lr': 1e-4,
    'updates': 6000,
    'eval_interval': 2000,
    'smoke_updates': 3,
    'dev_episodes': 2,
    'test_episodes': 8,
    'max_steps': 200,
    'seed': 42,
    'device': 'cuda',
    'github_repository': 'SongYunu/moveBoxes',
    'project_ref': 'main',
    'project_dir': '/content/moveBoxes_smol',
    'output_root': '/content/moveboxes_results',
    'env_root': '/content/moveboxes_envs_smol',
    'data_dir': '/content/moveboxes_rgb_data',
    'download_cache': '/content/moveboxes_data_cache',
    'simulator_repo': '/content/berlin-marso-foundation-smol',
    'simulator_commit': '6048f33217f26ae39009a812f53c81171517f393',
}

assert CFG['micro_batch'] * CFG['accumulate'] % 3 == 0
print('백엔드:', CFG['backend'], '· 실효 배치:', CFG['micro_batch'] * CFG['accumulate'])
print('한 모델에 Easy·Medium·Hard를 같은 비율로 넣습니다.')


In [ ]:
# 02 · GitHub 코드 로드 (기존 폴더가 있으면 최신 project_ref로 갱신)
import os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
if not (PROJECT/'.git').exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/SongYunu/moveBoxes.git',str(PROJECT)],check=True)
subprocess.run(['git','fetch','origin',CFG['project_ref']],cwd=PROJECT,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=PROJECT,check=True)
CFG['project_commit']=subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT,text=True).strip()
sys.path.insert(0,str(PROJECT))
from foundation.code.foundation_experiment import FoundationExperiment
experiment=FoundationExperiment(CFG)
print('사용 코드:',CFG['project_commit'])


In [ ]:
# 03 · GitHub 결과 복원/백업 연결
import getpass, os
if not os.environ.get('GH_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['GH_TOKEN']=userdata.get('GH_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub fine-grained token (Contents: read/write): ').strip()
        if not token: raise RuntimeError('결과 복원/저장용 GitHub token이 필요합니다.')
        os.environ['GH_TOKEN']=token
# 개인 비공개 사본에서는 위 분기 대신 os.environ['GH_TOKEN']='토큰'도 가능하지만 공개 커밋은 금지합니다.
experiment.connect()
experiment.report()


In [ ]:
# 04 · 모델/시뮬레이터 격리 환경 설치
experiment.install()


In [ ]:
# 05 · RGB 데이터 다운로드·해시 검증·600개 시연 분할
experiment.prepare()


In [ ]:
# 06 · T4 메모리/학습/정책/시뮬레이터 연결 실제 확인
# 이 셀이 성공하기 전에는 긴 학습을 시작하지 않습니다.
experiment.check()


In [ ]:
# 07 · Easy + Medium + Hard 공동 학습 (중간 최고 모델은 GitHub Release에 백업)
experiment.train()


In [ ]:
# 08 · 같은 최고 모델로 Easy 별도 테스트
easy_result=experiment.test('easy')


In [ ]:
# 09 · 같은 최고 모델로 Medium 별도 테스트
medium_result=experiment.test('medium')


In [ ]:
# 10 · 같은 최고 모델로 Hard 별도 테스트
hard_result=experiment.test('hard')


In [ ]:
# 11 · 공동 학습 이력과 세 난이도 결과 확인
experiment.report()
